In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from cuda_cqed.sim import Sim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Qt5Agg')
%matplotlib inline

# Hamiltonian

Here we are trying to replace Leon's code, so that we can add more features rapidly, like dephasing effects and the GPU acceleration.

I'll start in the rotating frame. There are many approximations that we may want to play with - the rotating frame/RWA being one. For very short pulses, hundreds of ns, it becomes tractable to simulate full dynamics - this has the nice effect of automatically including dynamic Kerr terms.

$$
H = \omega_b b^{\dagger} b + \omega_c c^{\dagger} c + 6 g_3 \lambda ( b^{\dagger} c \eta  + b c^{\dagger} \eta^* )  + K_4 b^{\dagger} b^{\dagger} b b + K_6 b^{\dagger} b^{\dagger} b^{\dagger} b b b
$$


No squeezing here, so throwing out the $bb$ type terms in the following

# Heisenberg EoM calcs

In [6]:
from openfermion.ops import BosonOperator
from openfermion.transforms import normal_ordered
from openfermion.utils import commutator

from sympy import symbols
import numpy as np
K4, K6, g3, η_1, conj_η_1, η_2, conj_η_2, η_3, conj_η_3 = symbols('K4, K6, g_3, η_1, conj_η_1, η_2, conj_η_2, η_3, conj_η_3', real=True)
wa, wb, wc, λ_ab, λ_bc = symbols('ω_a, ω_b, ω_c, λ_ab, λ_bc', real=True)

H_lin = BosonOperator('0^ 0', wa) + BosonOperator('1^ 1', wb) + BosonOperator('2^ 2', wc) 
H_stark = K4*BosonOperator('1^ 1^ 1 1') + K6*BosonOperator('1^ 1^ 1^ 1 1 1')
H_parametric = 6*g3*λ_ab*(conj_eta*BosonOperator('0^ 1')+eta*BosonOperator('0 1^')) + 6*g3*(conj_η_3*BosonOperator('1^ 1^')+η_3*BosonOperator('1 1')) + 6*g3*λ_bc*(conj_η_3*BosonOperator('1^ 2')+η_3*BosonOperator('1 2^')) 

H = H_lin + H_stark + H_parametric
print(H)

6.0*eta*g_3*λ_ab [0 1^] +
ω_a [0^ 0] +
6.0*conj_eta*g_3*λ_ab [0^ 1] +
6.0*g_3*η_3 [1 1] +
6.0*g_3*η_3*λ_bc [1 2^] +
ω_b [1^ 1] +
6.0*conj_η_3*g_3 [1^ 1^] +
1.0*K4 [1^ 1^ 1 1] +
1.0*K6 [1^ 1^ 1^ 1 1 1] +
6.0*conj_η_3*g_3*λ_bc [1^ 2] +
ω_c [2^ 2]


In [8]:
print('a EoM: ', normal_ordered(1j*commutator(H, BosonOperator('0'))))
print('')
print('aa EoM: ', normal_ordered(1j*commutator(H, BosonOperator('0 0'))))
print('')
print('adaga EoM: ', normal_ordered(1j*commutator(H, BosonOperator('0^ 0'))))
print('')
print('b EoM: ', normal_ordered(1j*commutator(H, BosonOperator('1'))))
print('')
print('bb EoM: ', normal_ordered(1j*commutator(H, BosonOperator('1 1'))))
print('')
print('bdagb EoM: ', normal_ordered(1j*commutator(H, BosonOperator('1^ 1'))))
print('')
print('c EoM: ', normal_ordered(1j*commutator(H, BosonOperator('2'))))
print('')
print('cc EoM: ', normal_ordered(1j*commutator(H, BosonOperator('2 2'))))
print('')
print('cdagc EoM: ', normal_ordered(1j*commutator(H, BosonOperator('2^ 2'))))
print('')
print('ab EoM: ', normal_ordered(1j*commutator(H, BosonOperator('0 1'))))
print('')
print('adagb EoM: ', normal_ordered(1j*commutator(H, BosonOperator('0^ 1'))))
print('')
print('bc EoM: ', normal_ordered(1j*commutator(H, BosonOperator('1 2'))))
print('')
print('bdagc EoM: ', normal_ordered(1j*commutator(H, BosonOperator('1^ 2'))))
print('')
print('ac EoM: ', normal_ordered(1j*commutator(H, BosonOperator('0 2'))))
print('')
print('adagc EoM: ', normal_ordered(1j*commutator(H, BosonOperator('0^ 2'))))

a EoM:  -1.0*I*ω_a [0] +
-6.0*I*conj_eta*g_3*λ_ab [1]

aa EoM:  -2.0*I*ω_a [0 0] +
-12.0*I*conj_eta*g_3*λ_ab [0 1]

adaga EoM:  6.0*I*eta*g_3*λ_ab [0 1^] +
-6.0*I*conj_eta*g_3*λ_ab [0^ 1]

b EoM:  -6.0*I*eta*g_3*λ_ab [0] +
-1.0*I*ω_b [1] +
-12.0*I*conj_η_3*g_3 [1^] +
-2.0*I*K4 [1^ 1 1] +
-3.0*I*K6 [1^ 1^ 1 1 1] +
-6.0*I*conj_η_3*g_3*λ_bc [2]

bb EoM:  -12.0*I*conj_η_3*g_3 [] +
-12.0*I*eta*g_3*λ_ab [0 1] +
-2.0*I*K4 - 2.0*I*ω_b [1 1] +
-12.0*I*conj_η_3*g_3*λ_bc [1 2] +
-24.0*I*conj_η_3*g_3 [1^ 1] +
-4.0*I*K4 - 6.0*I*K6 [1^ 1 1 1] +
-6.0*I*K6 [1^ 1^ 1 1 1 1]

bdagb EoM:  -6.0*I*eta*g_3*λ_ab [0 1^] +
6.0*I*conj_eta*g_3*λ_ab [0^ 1] +
12.0*I*g_3*η_3 [1 1] +
6.0*I*g_3*η_3*λ_bc [1 2^] +
-12.0*I*conj_η_3*g_3 [1^ 1^] +
-6.0*I*conj_η_3*g_3*λ_bc [1^ 2]

c EoM:  -6.0*I*g_3*η_3*λ_bc [1] +
-1.0*I*ω_c [2]

cc EoM:  -12.0*I*g_3*η_3*λ_bc [1 2] +
-2.0*I*ω_c [2 2]

cdagc EoM:  -6.0*I*g_3*η_3*λ_bc [1 2^] +
6.0*I*conj_η_3*g_3*λ_bc [1^ 2]

ab EoM:  -6.0*I*eta*g_3*λ_ab [0 0] +
-1.0*I*ω_a - 1.0*I*ω_b [0 1] +


# Cumulant equations


<!-- the Heisenberg EoMs are
$$
\dot{ <b> } = - i \omega_b <b> - 6 i \eta g_3 \lambda <c> - 2i K_4 <b^{\dagger} b > <b>
$$
$$
\dot{<b^{\dagger} b>} = +6 i \eta^* g_3 \lambda <b^{\dagger}c>^* - 6 i \eta g_3 \lambda <b^{\dagger}c>
$$

$$
\dot{ <c> } = - i \omega_c <c> - 6 i \eta^* g_3 \lambda <b>
$$
$$
\dot{<c^{\dagger} c>} = -6 i \eta^* g_3 \lambda <b^{\dagger}c>^* + 6 i \eta g_3 \lambda <b^{\dagger}c>
$$

$$
\dot{ <b^{\dagger} c> } = - 6i \eta^* g_3 \lambda <b^{\dagger} b> + i (\omega_b - \omega_c) <b^{\dagger}c> + 6 \eta^* g_3 \lambda <c^{\dagger} c> + 2i K_4 <b^{\dagger} c> <b^{\dagger} b>
$$


Adding the static loss terms
$$
\dot{ <b> } = - i \omega_b <b> - 6 i \eta g_3 \lambda <c> - 2i K_4 <b^{\dagger} b > <b> - \kappa_b/2 <b>
$$
$$
\dot{<b^{\dagger} b>} = +6 i \eta^* g_3 \lambda <b^{\dagger}c>^* - 6 i \eta g_3 \lambda <b^{\dagger}c>  - \kappa_b <b^{\dagger} b>
$$

$$
\dot{ <c> } = - i \omega_c <c> - 6 i \eta^* g_3 \lambda <b> - \kappa_c/2 <c>
$$
$$
\dot{<c^{\dagger} c>} = -6 i \eta^* g_3 \lambda <b^{\dagger}c>^* + 6 i \eta g_3 \lambda <b^{\dagger}c> - \kappa_c <c^{\dagger} c>
$$

$$
\dot{ <b^{\dagger} c> } = - 6i \eta^* g_3 \lambda <b^{\dagger} b> + i (\omega_b - \omega_c) <b^{\dagger}c> + 6 i \eta^* g_3 \lambda <c^{\dagger} c> + 2i K_4 <b^{\dagger} c> <b^{\dagger} b> - \frac{\kappa_b + \kappa_c}{2} <b^{\dagger} c>
$$
 -->

In [454]:
pi = np.pi

sim = Sim(use_complex=True)

sim.add_param('wb', 5.4e9*2*pi)
sim.add_param('wc', 7.3e9*2*pi, is_excitation=True)
sim.add_param('sqrtkb', np.sqrt(0.3e6 * 2 * np.pi))
sim.add_param('sqrtkc', np.sqrt(1.5e6 * 2 * np.pi))
sim.add_param('g3', 10e6*2*np.pi)
sim.add_param('K4', 0.1e6 * 2 * np.pi)
sim.add_param('K6', -0.01e6 * 2 * np.pi)
sim.add_param('lambdaa', 0.1)

sim.add_paramsweep('etaa', 0, 3, 101)
sim.add_paramsweep('wC2', 1.8e9 * 2 * np.pi,  1.95e9 * 2 * np.pi, 101)
sim.add_param('rampC2', 1e-9)5
sim.add_param('startC2', 20e-9)
sim.add_param('stopC2', 150e-9)
sim.add_param('phaseC2', 1)

sim.add_param('b_disp', 0.0)
sim.add_paramsweep('c_disp', 1, 10, 10)

C2pulse = sim.make_pulse('wC2', 'etaa', 'phaseC2', 'startC2', 'stopC2', 'rampC2')

sim.add_EOM('pump', C2pulse)
sim.add_EOM('a')
sim.add_EOM('aa')
sim.add_EOM('adaga')
sim.add_EOM('b')
sim.add_EOM('bb')
sim.add_EOM('bdagb')
sim.add_EOM('c')
sim.add_EOM('cc')
sim.add_EOM('cdagc')
sim.add_EOM('ab')
sim.add_EOM('adagb')
sim.add_EOM('bc')
sim.add_EOM('bdagc')
sim.add_EOM('ac')
sim.add_EOM('adagc')

# sim.add_EOM('b', '-1j*wb*b - 6j*conjugate(pump)*g3*lambdaa*c - sqrtkb**2/2*b - 2j*K4*bdagb*b - abs(K4)*bdagb*b  - 3j*K6*bdagb**2*b - abs(K6)*bdagb**2*b ', IC_str='b_disp')
# sim.add_EOM('bdagb', '6j*pump*g3*lambdaa*conjugate(bdagc) - 6j*conjugate(pump)*g3*lambdaa*bdagc - sqrtkb**2*bdagb', IC_str='abs(b_disp)**2')
# sim.add_EOM('c', '-1j*wc*c - 6j*pump*g3*lambdaa*b - sqrtkc**2/2*c', IC_str='c_disp')
# sim.add_EOM('cdagc', '- 6j*pump*g3*lambdaa*conjugate(bdagc) + 6j*conjugate(pump)*g3*lambdaa*bdagc - sqrtkc**2*cdagc', IC_str='abs(c_disp)**2')
# sim.add_EOM('bdagc', '-6j*pump*g3*lambdaa*bdagb + 1j*(wb-wc)*bdagc + 6j*pump*g3*lambdaa*cdagc + 2j*K4*bdagc*bdagb  + 3j*K6*bdagc*bdagb**2 - (sqrtkb**2/2+sqrtkc**2/2+abs(K4)*bdagb+abs(K6)*bdagb**2)*bdagc')
sim.set_solve_type('decimate')

sim.specify_time(pts_per_cycle=40, num_cycles=1000, d_factor=10)

sim.validate()

SyntaxError: invalid syntax (3298427655.py, line 16)

In [ ]:
x, t = sim.quick_trace()

scale = 10

plt.figure(1)
plt.clf()
plt.plot(t*1e9, x[0,:]/np.max(np.abs(x[0:1,:]))+4,color=(0.5,0.5,0.5,0.9),label='Re(pump)')
plt.plot(t*1e9, x[1,:]/np.max(np.abs(x[0:1,:]))+4,color=(0.5,0.5,1,0.5),label='Im(pump)')
plt.plot(t*1e9, x[2,:]/scale+2,color=(0,0,1,0.9),label=r'Re$\langle b \rangle $')
plt.plot(t*1e9, x[3,:]/scale+2,color=(0,1,0,0.5),label=r'Im$\langle b \rangle $')
plt.plot(t*1e9, np.sqrt(x[4,:])/scale+2,color=(1,0,1,0.5),label=r'$\sqrt{\langle b^{\dagger} b \rangle} $')
plt.plot(t*1e9, x[6,:]/scale+0,color=(0,0,1,0.9),label=r'Re$\langle c \rangle $')
plt.plot(t*1e9, x[7,:]/scale+0,color=(0,1,0,0.5),label=r'Im$\langle c \rangle $')
plt.plot(t*1e9, np.sqrt(x[8,:])/scale+0,color=(1,0,1,0.5),label=r'$\sqrt{\langle c^{\dagger} c \rangle} $')
plt.xlabel('Time (ns)')
plt.ylabel('Normalized amplitude')
plt.legend()
plt.yticks([0,2,4])
plt.grid()
plt.show()

Simulation validation success!
Running CPU quick solve...


In [ ]:
plt.plot(x[4,:],'b-')
plt.plot(x[5,:],'b--')

plt.plot(x[8,:],'g-')
plt.plot(x[9,:],'g--')

plt.plot(x[10,:],'r-', alpha=0.4)
plt.plot(x[11,:],'r--',alpha=0.4)
plt.show()

In [ ]:
I, Q, t = sim.solve()